# Stops recalibrated at 5 / 10 / 20 bp per side

Prior twelve months of saved trades select each threshold; the next three months use it unchanged. Equal utility for N=2 and N=3, one threshold shared by both. Models, forecasts, entries, weights, WFO, original observation masks and first-year warm-up are fixed. No-stop means hold the original position to close. This notebook inspects a completed release and cannot start fits or rebuild missing outputs. Results remain conditional on observed dates, with optimistic opening/stop fills and inherited research limitations.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").exists():
    PROJECT_DIR = PROJECT_DIR.parent
RELEASE = PROJECT_DIR / "data/backtests/14_stop_cost_recalibration_v1"
if not (RELEASE / "MANIFEST.json").exists():
    raise FileNotFoundError("Completed recalibration release required; inspection never builds or fits.")
manifest = json.loads((RELEASE / "MANIFEST.json").read_text())
assert manifest["complete"] and manifest["model_fits"] == 0
assert manifest["stop_calibration_bps_per_side"] == [5, 10, 20]
display(pd.DataFrame(manifest["model_catalog"]))
print(manifest["calibration_books"], "cost-specific calibration books;", manifest["original_1bp_controls"], "original-policy controls passed")

In [ ]:
metrics = pd.read_csv(RELEASE / "metrics.csv")
rich = metrics.loc[metrics.model.eq("ret_rich42")]
display(rich.pivot(index=["n_per_side", "approach"], columns="cost_bps_per_side", values="annual_compound"))
display(rich[["n_per_side", "approach", "cost_bps_per_side", "sessions", "sharpe", "max_drawdown", "stop_rate", "first_bar_fraction_of_stops", "mean_cost_bps", "mean_absolute_net_exposure_time"]])
display(Image(filename=str(RELEASE / "richer42_recalibrated_sensitivity.png")))
display(Image(filename=str(RELEASE / "richer42_recalibrated_equity.png")))

In [ ]:
comparison = pd.read_csv(RELEASE / "recalibrated_vs_frozen.csv")
display(comparison.loc[comparison.model.eq("ret_rich42"), ["n_per_side", "approach", "cost_bps_per_side", "annual_compound_recalibrated", "annual_compound_frozen_1bp", "cagr_change", "sharpe_recalibrated", "sharpe_frozen_1bp"]])
policies = pd.read_csv(RELEASE / "quarterly_stop_policies.csv")
display(policies.groupby(["model", "family", "cost_bps_per_side", "policy_status"]).size().rename("quarters").to_frame())
display(pd.read_parquet(RELEASE / "excluded_comparison_dates.parquet").groupby("model").size().rename("omitted_dates").to_frame())

## Execution sensitivity and shared dates

Some learned barriers can be very tight and can trigger in the opening bar. Fixed dollar fees do not establish executable prices, tick sizes, queue access, borrow or opening-input delivery. The following diagnostics retain those limitations. The nine-model table uses a common observed-date intersection for direct model comparisons; the bounded validation-only experiment stays separate.

In [ ]:
trades = pd.read_parquet(RELEASE / "ret_rich42_trades.parquet")
trades["distance_bps"] = 10000 * trades.distance_price / trades.entry_price
diagnostics = trades.groupby(["n_per_side", "approach", "cost_bps_per_side"]).agg(
    median_distance_bps=("distance_bps", "median"),
    minimum_distance_bps=("distance_bps", "min"),
    stop_rate=("stopped", "mean"),
    stops=("stopped", "sum"),
    first_bar_stops=("first_bar_stop", "sum"))
diagnostics["first_bar_fraction_of_stops"] = diagnostics.first_bar_stops / diagnostics.stops
display(diagnostics)
display(pd.read_csv(RELEASE / "nine_models_shared_dates_metrics.csv"))